# This file creates the master_customer_dataframe.ipynb

In [10]:
import pandas as pd

# Load individual and business customer data
individual_df = pd.read_csv('individual_cleaned_Jan16.csv')
business_df = pd.read_csv('business_cleaned_Jan16.csv')

# Load transaction datasets to get all customer_ids
abm_df = pd.read_csv('abm_cleaned.csv')
cheque_df = pd.read_csv('Cleaned_cheque.csv')
wire_df = pd.read_csv('Cleaned_wire.csv')
eft_df = pd.read_csv('eft.csv')
emt_df = pd.read_csv('emt.csv')
westernunion_df = pd.read_csv('westernunion.csv')

card_df = pd.read_csv('Card_Cleaned.csv')

In [11]:
# Get all unique customer_ids from all datasets (stable order)
print("Collecting unique customer_ids from all datasets...")

# Keep a fixed source order and preserve first appearance order.
# This makes row ordering deterministic across runs.
id_sources = [
    individual_df['customer_id'],
    business_df['customer_id'],
    abm_df['customer_id'],
    cheque_df['customer_id'],
    wire_df['customer_id'],
    eft_df['customer_id'],
    emt_df['customer_id'],
    westernunion_df['customer_id'],
    card_df['customer_id']
]

ordered_customer_ids = (
    pd.concat(id_sources, ignore_index=True)
    .dropna()
    .drop_duplicates(keep='first')
    .tolist()
)

print(f"Total unique customers: {len(ordered_customer_ids)}")
print(f"Customers in individual dataset: {individual_df['customer_id'].nunique()}")
print(f"Customers in business dataset: {business_df['customer_id'].nunique()}")

Total unique customers: 61410
Customers in individual dataset: 53099
Customers in business dataset: 8311


In [12]:
# Prepare individual customer data
print("Preparing individual customer data...")
individual_master = individual_df[['customer_id', 'country', 'province', 'city', 'occupation_code', 'income', 
                                    'birth_date', 'birth_age', 'onboard_date', 'account_age']].copy()
individual_master['kyc_type'] = 'individual'
individual_master['sales_cents'] = None
individual_master['industry_code'] = None

# Rename income to income_cents
individual_master = individual_master.rename(columns={'income': 'income_cents'})

# Create birth_establish_date from birth_date for individuals
individual_master['birth_establish_date'] = individual_master['birth_date']

# Create birth_business_age from birth_age for individuals
individual_master['birth_business_age'] = individual_master['birth_age']

# Drop the original birth_date and birth_age columns
individual_master = individual_master.drop(columns=['birth_date', 'birth_age'])

# Reorder columns
individual_master = individual_master[['customer_id', 'kyc_type', 'country', 'province', 'city', 
                                       'industry_code', 'occupation_code', 'income_cents', 'sales_cents',
                                       'account_age', 'onboard_date', 'birth_establish_date', 'birth_business_age']]

print(individual_master.head())

Preparing individual customer data...
       customer_id    kyc_type country province      city industry_code  \
0  SYNID0100000167  individual      CA       ON   TORONTO          None   
1  SYNID0100000431  individual      CA    other     other          None   
2  SYNID0100000485  individual      CA       ON  BRAMPTON          None   
3  SYNID0100000539  individual      CA    other     other          None   
4  SYNID0100000932  individual      CA       ON   TORONTO          None   

  occupation_code  income_cents sales_cents  account_age onboard_date  \
0           10019       4888600        None        14.28   2011-09-20   
1           72310            -1        None         7.58   2018-06-04   
2         RETIRED       1999800        None        28.47   1997-07-12   
3         RETIRED       3941700        None        40.64   1985-05-14   
4         RETIRED       3418200        None        13.48   2012-07-09   

  birth_establish_date  birth_business_age  
0           1972-01-30     

In [13]:
# Prepare business customer data
print("Preparing business customer data...")
business_master = business_df[['customer_id', 'country', 'province', 'city', 'industry_code', 'sales',
                               'established_date', 'business_age', 'onboard_date', 'account_age']].copy()
business_master['kyc_type'] = 'business'
business_master['income_cents'] = None
business_master['occupation_code'] = None

# Rename sales to sales_cents
business_master = business_master.rename(columns={'sales': 'sales_cents'})

# Create birth_establish_date from established_date for businesses
business_master['birth_establish_date'] = business_master['established_date']

# Create birth_business_age from business_age for businesses
business_master['birth_business_age'] = business_master['business_age']

# Drop the original established_date and business_age columns
business_master = business_master.drop(columns=['established_date', 'business_age'])

# Reorder columns to match individual_master
business_master = business_master[['customer_id', 'kyc_type', 'country', 'province', 'city', 
                                   'industry_code', 'occupation_code', 'income_cents', 'sales_cents',
                                   'account_age', 'onboard_date', 'birth_establish_date', 'birth_business_age']]

print(business_master.head())

Preparing business customer data...
       customer_id  kyc_type country province      city industry_code  \
0  SYNID0200000024  business      CA       ON  BRAMPTON          9699   
1  SYNID0200000050  business      CA       ON   TORONTO          4214   
2  SYNID0200000104  business      CA       AB  EDMONTON          7215   
3  SYNID0200000167  business      CA    other     other          8649   
4  SYNID0200000345  business      CA    other     other          9861   

  occupation_code income_cents  sales_cents  account_age onboard_date  \
0            None         None       181876         3.02   2022-12-26   
1            None         None       250009        29.54   1996-06-17   
2            None         None       217904          NaN          NaN   
3            None         None            0         1.10   2024-11-27   
4            None         None           -1        22.04   2003-12-18   

  birth_establish_date  birth_business_age  
0           2022-12-09                3.0

In [14]:
# Combine individual and business data
master_customers = pd.concat([individual_master, business_master], ignore_index=True)

print(f"Total customers in combined dataset: {len(master_customers)}")
print(f"Unique customers: {len(master_customers['customer_id'].unique())}")

# Check for duplicates
duplicates = master_customers[master_customers.duplicated(subset=['customer_id'], keep=False)]
if len(duplicates) > 0:
    print(duplicates.head(10))
else:
    print("\nNo duplicate customer_ids found")

Total customers in combined dataset: 61410
Unique customers: 61410

No duplicate customer_ids found


In [15]:
# Create a dataframe with all unique customer_ids (stable order)
all_customers_df = pd.DataFrame({'customer_id': ordered_customer_ids})

# Merge with the master_customers to get customer information
master_customer_df = all_customers_df.merge(
    master_customers, 
    on='customer_id', 
    how='left'
)

# Add sales_or_income column based on kyc_type
master_customer_df['sales_or_income'] = master_customer_df['kyc_type'].apply(
    lambda x: 'income' if x == 'individual' else ('sales' if x == 'business' else None)
)

print(f"Master customer dataframe created with {len(master_customer_df)} customers")
print(master_customer_df['kyc_type'].value_counts(dropna=False))
print(master_customer_df['sales_or_income'].value_counts(dropna=False))
print(f"\nDataFrame shape: {master_customer_df.shape}")

Master customer dataframe created with 61410 customers
kyc_type
individual    53099
business       8311
Name: count, dtype: int64
sales_or_income
income    53099
sales      8311
Name: count, dtype: int64

DataFrame shape: (61410, 14)


In [16]:
# Reorder columns for better readability
column_order = ['customer_id', 'kyc_type', 'country', 'province', 'city', 
                'industry_code', 'occupation_code', 'sales_or_income', 'income_cents', 'sales_cents',
                'account_age', 'onboard_date', 'birth_establish_date', 'birth_business_age']

master_customer_df = master_customer_df[column_order]

print("Final master customer dataframe:")
print(master_customer_df.head(10))

Final master customer dataframe:
       customer_id    kyc_type country province      city industry_code  \
0  SYNID0100000167  individual      CA       ON   TORONTO          None   
1  SYNID0100000431  individual      CA    other     other          None   
2  SYNID0100000485  individual      CA       ON  BRAMPTON          None   
3  SYNID0100000539  individual      CA    other     other          None   
4  SYNID0100000932  individual      CA       ON   TORONTO          None   
5  SYNID0100001185  individual      CA       AB  EDMONTON          None   
6  SYNID0100001300  individual      CA       AB   CALGARY          None   
7  SYNID0100001310  individual      CA    other     other          None   
8  SYNID0100001952  individual      CA       ON   SUDBURY          None   
9  SYNID0100002045  individual      CA       ON  KINGSTON          None   

  occupation_code sales_or_income income_cents sales_cents  account_age  \
0           10019          income      4888600        None        

In [17]:
# Summary statistics
print("Summary Statistics:")
print("=" * 50)
print(master_customer_df['kyc_type'].value_counts(dropna=False))
print(master_customer_df['sales_or_income'].value_counts(dropna=False))
print(master_customer_df['country'].value_counts(dropna=False).head(10))
print(master_customer_df['province'].value_counts(dropna=False).head(10))
print(f"\nCustomers with income data: {master_customer_df['income_cents'].notna().sum()}")
print(f"Customers with sales data: {master_customer_df['sales_cents'].notna().sum()}")

Summary Statistics:
kyc_type
individual    53099
business       8311
Name: count, dtype: int64
sales_or_income
income    53099
sales      8311
Name: count, dtype: int64
country
CA    61410
Name: count, dtype: int64
province
ON       24130
other    20317
AB        5620
BC        5250
QC        1698
MB        1511
NS        1008
NB         729
NL         555
SK         494
Name: count, dtype: int64

Customers with income data: 53099
Customers with sales data: 8311


In [18]:
# Sort by customer id and Save the master customer dataframe
master_customer_df = master_customer_df.sort_values('customer_id', ascending=True).reset_index(drop=True)
master_customer_df.to_csv('master_customer_dataframe.csv', index=False)
print(f"File contains {len(master_customer_df)} customers with {len(master_customer_df.columns)} columns")

File contains 61410 customers with 14 columns
